In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import itertools
import re
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize, ListedColormap, BoundaryNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from HelperFunctions import detect_turns
%load_ext rpy2.ipython

In [ ]:
sns.set_theme(style="ticks", context="talk", font="Arial")

plt.rcParams.update({
    # Font + text sizes
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],  # fallbacks

    # Line widths
    "axes.linewidth": 0.5,   
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2,     
    "ytick.major.size": 2,

    # --- Keep text editable in exports ---
    "svg.fonttype": "none",   # SVG: keep text as <text> elements, not paths
    "pdf.fonttype": 42,       # PDF: embed fonts as TrueType, editable
    "ps.fonttype": 42,        # EPS: same as above
    "text.usetex": False,     # don’t force LaTeX (avoids path-conversion)
})

In [ ]:
df = pd.read_csv(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\ResidualTrialDF_Reward.csv')
df['Session'] = np.where(df['Session'] == 'Reward3', 'Reward', 'Monster')
df["Session"] = pd.Categorical(df["Session"], categories=['Reward', 'Monster'], ordered=True)

successdf = df.groupby(['Animal', 'Session', 'Trial'], observed=True).first().reset_index()[['Animal', 'Session', 'Trial', 'Time to Reward', 'Time to Monster']]
successdf['Rewarded'] = np.where(successdf['Time to Reward'].notna(), 1, 0)
successdf = successdf.groupby(['Animal', 'Session'], observed=True).agg("mean").reset_index()
goodanimals  = successdf[(successdf['Session'] == 'Reward') & (successdf['Rewarded'] >= 0.8)]['Animal'].to_list()

df = df[df['Animal'].isin(goodanimals)]
df['Site'] = df['Region']
df['Region'] = df['Site'].str[:2]

df = (
    df
      .groupby(['Animal','Session','Trial','Region'], observed=True)
      .apply(detect_turns, sweep_thresh_deg=60)
      .reset_index(drop=True)
)

#IpsiContra
df.loc[df["Site"] == "TSR", "Heading_body"] *= -1
df.loc[
    (df["Site"] == "VS") & (~df["Animal"].str.contains("EI")),
    "Heading_body"] *= -1

df['dtheta'] = (
    df.groupby(['Animal','Session','Trial','Region'], observed=True)['Heading_body']
       .diff()
)

df['ang_vel'] = df['dtheta'] / df['dt']

#IpsiContra
df.loc[df["Site"] == "TSR", "heading_relative_to_reward"] *= -1
df.loc[
    (df["Site"] == "VS") & (~df["Animal"].str.contains("EI")),
    "heading_relative_to_reward"] *= -1

df['dtheta'] = (
    df.groupby(['Animal','Session','Trial','Region'], observed=True)['heading_relative_to_reward']
       .diff()
)

df['ang_vel_reward'] = df['dtheta'] / df['dt']

prev = df['Turn'].shift(fill_value=0)
df['Turn Start'] = df['Turn'].where((df['Turn'] != 0) & (prev == 0), 0)
df.columns = df.columns.str.replace(" ", ".", regex=False)

df['Heading_deg'] = df['Heading_body'] % 360

low, high = df['velocity'].quantile([0.01, 0.99], interpolation="linear")

# keep only rows within [low, high]
mask = df['velocity'].between(low, high, inclusive="both")
removed = (~mask).sum()
df = df.loc[mask].copy()

df = df[df['X']>0]

#Transform radially projected velocities and accelerations into intuitive values (+fast and towards, - fast and away)
df['v_reward_radial'] = df['v_reward_radial']*-1
df['a_reward_radial'] = df['a_reward_radial']*-1

df = df[~((df['Region']=='TS')&(df['Animal']=='Wanda'))].copy()
df = df[~((df['Site']=='TSR')&(df['Animal']=='Chester'))].copy()

Figure 4C

In [ ]:
def plot_forest(
    res: pd.DataFrame,
    # columns
    estimate_col: str = "Estimate",
    se_col: str = "Std. Error",
    lo_col: str | None = None,
    hi_col: str | None = None,
    term_col: str | None = None,    # if None, parsed from index/first col
    group_col: str | None = None,   # if None, auto-uses parsed Region
    # parsing
    region_prefix: str = "Region",
    strip_scale: bool = True,
    z_crit: float = 1.96,
    # ordering
    order: list[str] | None = None,       # by DISPLAY labels (after rename)
    order_raw: list[str] | None = None,   # by RAW names (before rename)
    order_by: str | None = None,          # None | "estimate" | "abs" | "alpha"
    ascending: bool = False,
    # style
    figsize: tuple = (7, 8),
    xlabel: str = "Effect size (estimate ± 95% CI)",
    ylabel: str = "",
    tick_fs: float = 10,
    ytick_linespacing: float = 1.1,
    label_fs: float = 12,
    title: str | None = None,
    zero_ls: str = "--",
    zero_lw: float = 1.0,
    line_lw: float = 2.0,
    capsize: float = 5,
    point_ms: float = 6.0,
    dodge: float = 0.25,
    spine_color: str = "black",
    # legend
    legend_title: str | None = "",
    legend_loc: str = "upper left",
    legend_bbox: tuple | None = (1.1, 1.3),
    legend_fs: float = 9,
    legend_markerscale: float = 0.85,
    legend_borderaxespad: float = 0.3,
    # labeling & colors
    term_rename: dict | None = None,
    group_rename: dict | None = None,
    group_colors: dict | None = None,
    ax=None
):
    df = res.copy()
    if term_col and term_col in df.columns:
        termfull = df[term_col].astype(str)
    else:
        likely = [c for c in df.columns if c.lower() in {"term", "termfull", "coefficient", "coef"}]
        if likely:
            termfull = df[likely[0]].astype(str)
        else:
            df = df.reset_index()
            termfull = df.iloc[:, 0].astype(str)
    df["TermFull"] = termfull

    pat = rf":{re.escape(region_prefix)}([A-Za-z0-9_]+)$"
    reg = df["TermFull"].str.extract(pat, expand=False)
    term_raw = df["TermFull"].str.replace(pat, "", regex=True)
    if strip_scale:
        term_raw = term_raw.str.replace(r"^scale\((.*)\)$", r"\1", regex=True)
    df["Term_raw"] = term_raw
    df["Term"] = df["Term_raw"].map(lambda t: term_rename.get(t, t) if term_rename else t)
    if group_col is None and reg.notna().any():
        df["Region"] = reg
        group_col = "Region"
    if group_col is not None and group_col not in df.columns:
        df[group_col] = reg

    if group_col is not None:
        disp_group = df[group_col].map(lambda g: group_rename.get(g, g) if group_rename else g).astype(str)
        df["_group_label"] = disp_group
        groups = pd.unique(disp_group).tolist()
    else:
        df["_group_label"] = None
        groups = [None]

    need_ci = (not lo_col or lo_col not in df.columns) or (not hi_col or hi_col not in df.columns)
    _lo = lo_col if (lo_col and lo_col in df.columns) else "_lo"
    _hi = hi_col if (hi_col and hi_col in df.columns) else "_hi"
    if need_ci:
        if se_col not in df.columns:
            raise ValueError("Provide lo/hi or a SE column to compute CIs.")
        df[_lo] = df[estimate_col] - z_crit * df[se_col]
        df[_hi] = df[estimate_col] + z_crit * df[se_col]

    if order_raw is not None:
        raw2disp = df.drop_duplicates("Term_raw").set_index("Term_raw")["Term"].to_dict()
        levels = [raw2disp.get(r, r) for r in order_raw]
    elif order is not None:
        levels = list(order)
    else:
        if order_by is None:
            levels = pd.unique(df["Term"]).tolist()
        elif order_by == "estimate":
            levels = df.groupby("Term", observed=True)[estimate_col].mean().sort_values(ascending=ascending).index.tolist()
        elif order_by == "abs":
            levels = df.groupby("Term", observed=True)[estimate_col].apply(lambda s: s.abs().mean()).sort_values(ascending=ascending).index.tolist()
        elif order_by == "alpha":
            levels = sorted(pd.unique(df["Term"]).tolist(), reverse=not ascending)
        else:
            raise ValueError("order_by must be None|'estimate'|'abs'|'alpha'.")

    ymap = {t: i for i, t in enumerate(levels[::-1])}
    df["_y"] = df["Term"].map(ymap)

    if group_col is not None:
        center = (len(groups) - 1) / 2
        offsets = {g: (i - center) * dodge for i, g in enumerate(groups)} if len(groups) > 1 else {groups[0]: 0.0}
        df["_y_off"] = df["_group_label"].map(offsets).fillna(0.0)
    else:
        df["_y_off"] = 0.0


    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        created_fig = True
    else:
        fig = ax.figure

    ax.axvline(0, ls=zero_ls, lw=zero_lw, color="0.4", zorder=0)

    default_cycle = plt.rcParams["axes.prop_cycle"].by_key().get("color", [])
    def pick_color(lbl, i):
        if group_colors and lbl in group_colors: 
            return group_colors[lbl]
        return default_cycle[i % len(default_cycle)] if default_cycle else None

    handles, labels = [], []
    for gi, g in enumerate(groups):
        sub = df if g is None else df[df["_group_label"] == g]
        if sub.empty:
            continue
        y = sub["_y"] + sub["_y_off"]
        color = pick_color(g, gi)
        h = ax.hlines(y, sub[_lo], sub[_hi], lw=line_lw, color=color, alpha=0.95,
                      label=(str(g) if g is not None else None))
        ax.errorbar(sub[estimate_col], y,
                    xerr=[sub[estimate_col] - sub[_lo], sub[_hi] - sub[estimate_col]],
                    fmt="o", ms=point_ms, lw=0, capsize=capsize, color=color)
        if g is not None:
            handles.append(h); labels.append(str(g))

    yt = [ymap[t] for t in levels[::-1]]
    ax.set_yticks(yt)
    ax.set_yticklabels(levels[::-1], fontsize=tick_fs)
    for t in ax.get_yticklabels():
        t.set_multialignment('center')
        try:
            t.set_verticalalignment('center_baseline')
        except Exception:
            t.set_verticalalignment('center')
        t.set_linespacing(ytick_linespacing)
    ax.yaxis.tick_right(); ax.yaxis.set_label_position("right")

    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_visible(True);  ax.spines["right"].set_color(spine_color); ax.spines["right"].set_linewidth(1)
    ax.spines["bottom"].set_visible(True); ax.spines["bottom"].set_color(spine_color); ax.spines["bottom"].set_linewidth(1)
    ax.spines["top"].set_visible(False)

    ax.set_xlabel(xlabel, fontsize=label_fs)
    ax.set_ylabel(ylabel, fontsize=label_fs)
    if title:
        ax.set_title(title, fontsize=label_fs)

    if group_col is not None and labels:
        ax.legend(
            handles, labels,
            title=(legend_title or ""),
            frameon=False,
            loc=legend_loc,
            bbox_to_anchor=legend_bbox,
            fontsize=legend_fs,
            markerscale=legend_markerscale,
            borderaxespad=legend_borderaxespad,
        )

    ax.tick_params(axis="both", labelsize=tick_fs)

    if created_fig:
        try:
            fig.tight_layout(rect=(0, 0, 0.86, 1))
        except Exception:
            pass

    return fig, ax

In [ ]:
# 1 sec bins to deal with autocorrelation

g = ["Region","Session","Animal","Trial","Site"]
k = np.where(np.isclose(df.dt, 1/12, atol=1e-9), 12, 20)

num = [c for c in df.select_dtypes("number").columns if c not in g]
obj = [c for c in df.columns if c not in num+g]

df = (df.assign(_k=k, _i=df.groupby(g).cumcount(), bin=lambda d: d._i//d._k)
        .groupby(g+["bin"], as_index=False, observed=True)
        .agg({**{c:"mean" for c in num}, **{c:"first" for c in obj}})
        .drop(columns=["bin"])
        .assign(Z_lag1=lambda d: d.groupby(g)["Z"].shift(1))
     )

In [ ]:
%%R -i df -o res_movement_model -o res_full_model

library(lme4)
library(emmeans)
library(performance)

df$Animal <- as.factor(df$Animal)
df$Session <- as.factor(df$Session)
df$Site <- as.factor(df$Site)
df$Region <- as.factor(df$Region)
df$Trial <- as.factor(df$Trial)

movement_model <- lmer(
  Z ~ ( scale(speed) + scale(acceleration) + scale(ang_vel) + scale(Z_lag1)) : Region - 1 + (1 | Animal) + (1 | Animal:Site), data = df, REML = FALSE)
acf(resid(movement_model))

m7_plus_all <- lmer( 
    Z ~ (scale(Distance.to.Spout) + scale(speed) + scale(acceleration) + scale(ang_vel) + scale(v_reward_radial) + scale(a_reward_radial) + scale(ang_vel_reward) + scale(Z_lag1)) : Region - 1 + (1 | Animal) + (1 | Animal:Site), data = df, REML = FALSE)
acf(resid(m7_plus_all))


res_movement_model <- as.data.frame(summary(movement_model)$coefficient)
res_full_model <- as.data.frame(summary(m7_plus_all)$coefficient)
print(anova(m7_plus_all, movement_model))

print(compare_performance(movement_model, m7_plus_all))

print(AIC(movement_model) - AIC(m7_plus_all))
print(BIC(movement_model) - BIC(m7_plus_all))

In [ ]:
fig, ax_forest = plt.subplots(figsize=(10, 4))

term_map  = {"Distance.to.Spout":"Distance to Spout", "speed":"Speed", "acceleration":"Acceleration", "ang_vel":"Angular Velocity",
                'v_reward_radial': 'Radial Velocity', 'a_reward_radial': 'Radial Acceleration',
                'ang_vel_reward' : 'Angular Velocity to Spout'}
colors    = {'VS': '#1f77b4', 'TS': '#ff7f0e'}
order_raw = ["speed","acceleration","ang_vel", "Distance.to.Spout", 'v_reward_radial', 'a_reward_radial', 'ang_vel_reward']

plot_forest(res_full_model, term_rename=term_map, group_colors=colors, order_raw=order_raw,
            legend_bbox=(1, 1.3), ax=ax_forest)

fig.savefig(r'D:\Uchida Lab Dropbox\Eshaan Iyer\Isobel_Eshaan\Code\Figure Mockup\SVGs\BinnedForestPlot.svg')